### LunaSurface AI - Lunar Surface Hazard Terrain Segmentation with Self-Supervised Learning

#### Imports & Performance Setup

In [ ]:
import os
import time
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

torch.set_float32_matmul_precision("medium")

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print("Using device:", DEVICE)

#### Config

In [ ]:
UNLABELED_DIR = "/Users/pheonix/Documents/Minor Project/LunaSurface-AI/Backend/Moon_Data/patches"
LABELED_IMG_DIR = "/Users/pheonix/Documents/Minor Project/LunaSurface-AI/Backend/Moon_Data/1.LabeledDataset/images"
LABELED_MASK_DIR = "/Users/pheonix/Documents/Minor Project/LunaSurface-AI/Backend/Moon_Data/1.LabeledDataset/masks"

IMG_SIZE = 512
BATCH_SIZE = 6
EPOCHS_SSL = 8
EPOCHS_FINE = 8

NUM_CLASSES = 4
BLOCK_SIZE = 64
MASK_RATIO = 0.4

In [ ]:
class UnlabeledDataset(Dataset):
    def __init__(self, folder, max_images=None, seed=42):
        self.paths = []

        # Collect all PNG image paths
        for root, _, files in os.walk(folder):
            for f in files:
                if f.lower().endswith(".png"):
                    self.paths.append(os.path.join(root, f))

        # 🔥 Random Subset Selection
        if max_images is not None:
            np.random.seed(seed)              # reproducible
            np.random.shuffle(self.paths)     # shuffle
            self.paths = self.paths[:max_images]

        print(f"✅ SSL Dataset Loaded: {len(self.paths)} images")

        self.tf = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.Grayscale(1),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx])
        return self.tf(img)


# 🔥 Choose how many images you want
MAX_SSL_IMAGES = 100   # change this number anytime

unlabeled_dataset = UnlabeledDataset(
    UNLABELED_DIR,
    max_images=MAX_SSL_IMAGES,
    seed=42
)

unlabeled_loader = DataLoader(
    unlabeled_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

In [ ]:
class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32,32,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # 512→256

            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64,64,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2),  # 256→128

            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128,128,3,padding=1), nn.ReLU()
        )

    def forward(self,x):
        return self.model(x)

In [ ]:
class Decoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.ConvTranspose2d(128,64,2,stride=2), nn.ReLU(),
            nn.ConvTranspose2d(64,1,2,stride=2)
        )

    def forward(self,x):
        return self.model(x)

In [ ]:
def block_mask(x, mask_ratio=MASK_RATIO, block_size=BLOCK_SIZE):

    B,C,H,W = x.shape
    x_masked = x.clone()

    num_blocks_h = H // block_size
    num_blocks_w = W // block_size
    total_blocks = num_blocks_h * num_blocks_w
    num_mask = int(total_blocks * mask_ratio)

    for b in range(B):
        perm = torch.randperm(total_blocks, device=x.device)[:num_mask]

        for idx in perm:
            row = idx // num_blocks_w
            col = idx % num_blocks_w

            h = row * block_size
            w = col * block_size

            x_masked[b,:,h:h+block_size,w:w+block_size] = 0

    return x_masked

In [ ]:
import sys
import time

encoder = Encoder().to(DEVICE)
decoder = Decoder().to(DEVICE)

optimizer = torch.optim.Adam(
    list(encoder.parameters()) + list(decoder.parameters()),
    lr=1e-3
)

loss_fn_ssl = nn.MSELoss()

GREEN = "\033[92m"
RESET = "\033[0m"

for epoch in range(EPOCHS_SSL):

    encoder.train()
    decoder.train()

    start_time = time.time()
    total_loss = 0
    total_batches = len(unlabeled_loader)

    print(f"\nEpoch {epoch+1}/{EPOCHS_SSL}")

    for step, imgs in enumerate(unlabeled_loader):

        imgs = imgs.to(DEVICE)
        masked = block_mask(imgs)

        feats = encoder(masked)
        recon = decoder(feats)

        loss = loss_fn_ssl(recon, imgs)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        progress = (step + 1) / total_batches
        percent = progress * 100

        bar_length = 30
        filled = int(bar_length * progress)

        # 🔥 Double line style
        bar = "=" * filled + " " * (bar_length - filled)

        sys.stdout.write(
            f"\r{GREEN}[{bar}] {percent:6.2f}% | loss: {loss.item():.4f}{RESET}"
        )
        sys.stdout.flush()

    epoch_time = time.time() - start_time
    avg_loss = total_loss / total_batches

    print(f"\nCompleted in {epoch_time:.0f}s "
          f"- Avg loss: {avg_loss:.4f}")

In [ ]:
torch.save(encoder.state_dict(), "ssl_encoder_advanced.pth")
print("SSL Encoder Saved")

In [ ]:
encoder = Encoder().to(DEVICE)
encoder.load_state_dict(torch.load("ssl_encoder_advanced.pth"))

for p in encoder.parameters():
    p.requires_grad = False

In [ ]:
class LabeledDataset(Dataset):
    def __init__(self, img_dir, mask_dir):

        self.images = sorted(os.listdir(img_dir))
        self.img_dir = img_dir
        self.mask_dir = mask_dir

        self.tf_img = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.Grayscale(1),
            transforms.ToTensor()
        ])

        self.tf_mask = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE),
                              interpolation=Image.NEAREST),
            transforms.PILToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        name = self.images[idx]

        img = Image.open(os.path.join(self.img_dir, name))
        mask = Image.open(os.path.join(self.mask_dir, name))

        img = self.tf_img(img)
        mask = self.tf_mask(mask).long().squeeze(0)

        return img, mask


dataset_full = LabeledDataset(LABELED_IMG_DIR, LABELED_MASK_DIR)

train_size = int(0.8 * len(dataset_full))
val_size = len(dataset_full) - train_size

train_set, val_set = random_split(dataset_full, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, num_workers=0)

In [ ]:
class SegHead(nn.Module):
    def __init__(self):
        super().__init__()

        self.head = nn.Sequential(
            nn.Conv2d(128,128,3,padding=1),
            nn.ReLU(),
            nn.Conv2d(128,NUM_CLASSES,1)
        )

    def forward(self,x):
        return self.head(x)

In [ ]:
class SegModel(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = SegHead()

    def forward(self,x):

        feats = self.encoder(x)
        mask = self.head(feats)

        mask = F.interpolate(
            mask,
            size=(IMG_SIZE, IMG_SIZE),
            mode="bilinear",
            align_corners=False
        )

        return mask

seg_model = SegModel(encoder).to(DEVICE)

In [ ]:
weights = torch.tensor([0.1,1,1,1]).to(DEVICE)
ce_loss = nn.CrossEntropyLoss(weight=weights)

def dice_loss(pred, target, eps=1e-6):

    pred = F.softmax(pred, dim=1)
    target_onehot = F.one_hot(target, NUM_CLASSES).permute(0,3,1,2).float()

    intersection = (pred * target_onehot).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target_onehot.sum(dim=(2,3))

    dice = (2*intersection + eps) / (union + eps)

    return 1 - dice.mean()

optimizer = torch.optim.Adam(seg_model.head.parameters(), lr=1e-4)

In [ ]:
for epoch in range(EPOCHS_FINE):

    seg_model.train()
    train_loss = 0

    for imgs, masks in train_loader:

        imgs = imgs.to(DEVICE)
        masks = masks.to(DEVICE)

        preds = seg_model(imgs)

        loss = ce_loss(preds, masks) + dice_loss(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # validation
    seg_model.eval()
    val_loss = 0

    with torch.no_grad():
        for imgs, masks in val_loader:

            imgs = imgs.to(DEVICE)
            masks = masks.to(DEVICE)

            preds = seg_model(imgs)

            loss = ce_loss(preds, masks) + dice_loss(preds, masks)
            val_loss += loss.item()

    print(f"Epoch {epoch} Train {train_loss/len(train_loader)} Val {val_loss/len(val_loader)}")

In [ ]:
seg_model.eval()

img, mask = next(iter(val_loader))
img = img.to(DEVICE)

with torch.no_grad():
    pred = seg_model(img)
    pred_class = torch.argmax(F.softmax(pred, dim=1), dim=1)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.title("Image")
plt.imshow(img[0,0].cpu(), cmap="gray")

plt.subplot(1,3,2)
plt.title("Ground Truth")
plt.imshow(mask[0], cmap="jet")

plt.subplot(1,3,3)
plt.title("Prediction")
plt.imshow(pred_class[0].cpu(), cmap="jet")

plt.show()

In [ ]:
torch.save(seg_model.state_dict(),
           "lunar_ssl_segmentation_advanced.pth")

print("Final Model Saved")